# Experiment - 5-fold cross-validation and weight tuning

Replaces the hand-picked ensemble weight with a grid search over out-of-fold slide probabilities, obtained from a slide-level stratified 5-fold split.

**Test F1: 0.4234.** This is also the notebook that produced `artifacts/oof_slide_probs_5fold.npz`, the file the reproducible scripts in this repository read.

---


## Setup and paths

Originally executed on Google Colab with the dataset on Google Drive. The mount
has been replaced by a portable path configuration: point `WSI_DATA_DIR` at a
directory holding the competition data (see `data/README.md`).


In [ ]:
# --- Paths ---------------------------------------------------------------
# Originally executed on Google Colab with the dataset on Google Drive.
# DATA_DIR must contain train_data/, test_data/ and train_labels.csv
# (see data/README.md).
import os

DATA_DIR = os.environ.get("WSI_DATA_DIR", "data")
os.makedirs("models", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)


## Goal and imports

Trains both branches of the final ensemble under a single slide-level 5-fold
split, tunes the mixing weight on out-of-fold predictions, refits both models on
all training slides, and produces the submitted predictions.


In [2]:
# 5-fold CV (slide-level) for:
# - ResNet18 tile classifier -> slide mean probs
# - UNI frozen backbone + MLP classifier on UNI embeddings -> slide mean probs
# Then: OOF probs -> best alpha for ensemble
# Finally: full training (ResNet18 + UNI-MLP) and test submission with best alpha

import os, random, math, json
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix

torch.set_float32_matmul_precision("high")

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


## Reproducibility and device

Seeds Python, NumPy and PyTorch. Note that this pins the data order and the
initialisation, not the arithmetic: cuDNN kernel selection still makes a GPU
re-run differ slightly.


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## Loading the preprocessed tiles

Loads the tile arrays produced by `01_preprocessing_and_tiles.ipynb`. Each tile
is 256x256, carries the label of the slide it came from, and keeps a reference to
that slide so predictions can be pooled back per slide.


In [4]:
def load_npz(path):
    d = np.load(path, allow_pickle=True)
    return d["tiles"], d["labels"], d["slide_ids"]

train_tiles, train_labels, train_slide_ids = load_npz("train_tiles_raw.npz")
test_tiles, _, test_slide_ids = load_npz("test_tiles_raw_3.npz")

print("TRAIN tiles:", train_tiles.shape, train_tiles.dtype)
print("TRAIN labels:", train_labels.shape, train_labels.dtype, "unique:", np.unique(train_labels))
print("TRAIN slide_ids:", train_slide_ids.shape, type(train_slide_ids[0]), "ex:", train_slide_ids[:3])

print("TEST tiles:", test_tiles.shape, test_tiles.dtype)
print("TEST slide_ids:", test_slide_ids.shape, type(test_slide_ids[0]), "ex:", test_slide_ids[:3])

TRAIN tiles: (3392, 256, 256, 3) uint8
TRAIN labels: (3392,) int8 unique: [0 1 2 3]
TRAIN slide_ids: (3392,) <class 'numpy.str_'> ex: ['img_0000.png' 'img_0000.png' 'img_0000.png']
TEST tiles: (2547, 256, 256, 3) uint8
TEST slide_ids: (2547,) <class 'numpy.str_'> ex: ['img_0000.png' 'img_0000.png' 'img_0000.png']


## Slide index

Groups tiles by slide. This index is what makes the cross-validation honest:
splits are drawn over *slides*, never over tiles, so no slide contributes tiles
to both sides of a fold.


In [5]:
slide_to_idxs = defaultdict(list)
slide_to_label = {}

for i, sid in enumerate(train_slide_ids):
    slide_to_idxs[sid].append(i)
    slide_to_label[sid] = int(train_labels[i])

train_slides = np.array(sorted(slide_to_idxs.keys()))
train_slide_y = np.array([slide_to_label[s] for s in train_slides], dtype=np.int64)

print("n_train_slides:", len(train_slides))
print("slide label counts:", np.bincount(train_slide_y, minlength=4))
print("tiles/slide min/median/mean/max:",
      np.min([len(slide_to_idxs[s]) for s in train_slides]),
      np.median([len(slide_to_idxs[s]) for s in train_slides]),
      np.mean([len(slide_to_idxs[s]) for s in train_slides]),
      np.max([len(slide_to_idxs[s]) for s in train_slides]))

n_train_slides: 627
slide label counts: [174 219 158  76]
tiles/slide min/median/mean/max: 1 5.0 5.409888357256778 27


## Label maps

The four molecular subtypes and their integer encoding. The same mapping is used
when writing the submission, so an inconsistency here would silently permute
every prediction.


In [6]:
IDX_TO_LABEL = {
    0: "Luminal A",
    1: "Luminal B",
    2: "HER2(+)",
    3: "Triple negative"
}
LABEL_TO_IDX = {v: k for k, v in IDX_TO_LABEL.items()}

## Augmentation and normalisation for ResNet18

Flips and 90-degree rotations only. Tissue tiles have no canonical orientation,
so the dihedral group is label-preserving here — the same symmetry the
test-time augmentation exploits later.


In [19]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3,1,1)

def normalize_imagenet(x_chw_float01):
    return (x_chw_float01 - IMAGENET_MEAN) / IMAGENET_STD

def aug_train_tensor(x_in):
    x = x_in  # explicit local binding (important for DataLoader workers)

    if random.random() < 0.5:
        x = torch.flip(x, dims=[2])  # horizontal
    if random.random() < 0.5:
        x = torch.flip(x, dims=[1])  # vertical

    k = random.randint(0, 3)
    if k != 0:
        x = torch.rot90(x, k, dims=[1, 2])

    return normalize_imagenet(x)

def aug_val_tensor(x_chw_float01):
    return normalize_imagenet(x_chw_float01)

## Datasets for ResNet18


In [8]:
class TileDataset(Dataset):
    def __init__(self, tiles_uint8_hwc, labels_int=None, idxs=None, train_aug=False):
        self.tiles = tiles_uint8_hwc
        self.labels = labels_int
        self.idxs = np.arange(len(self.tiles)) if idxs is None else np.array(idxs)
        self.train_aug = train_aug

    def __len__(self):
        return len(self.idxs)

    def __getitem__(self, i):
        idx = self.idxs[i]
        x = torch.from_numpy(self.tiles[idx]).permute(2,0,1).float() / 255.0  # CHW float01
        x = aug_train_tensor(x) if self.train_aug else aug_val_tensor(x)
        if self.labels is None:
            return x
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y

## Class weights and the ResNet18 training loop

Class weights are computed at tile level and normalised to mean 1. With label
smoothing at 0.1 and AdamW, this is the whole regularisation budget: the model is
small and the training set is 627 slides.


In [9]:
def compute_class_weights_from_tile_labels(y_tiles, n_classes=4):
    counts = np.bincount(y_tiles, minlength=n_classes).astype(np.float32)
    w = (counts.sum() / np.clip(counts, 1.0, None))
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

def train_resnet18_on_fold(train_tile_idxs, y_tiles_full, epochs=15, bs=64, lr=1e-4, wd=1e-2, label_smoothing=0.1):
    y_fold = y_tiles_full[train_tile_idxs].astype(np.int64)
    weights = compute_class_weights_from_tile_labels(y_fold, n_classes=4)

    ds = TileDataset(train_tiles, train_labels, idxs=train_tile_idxs, train_aug=True)
    dl = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True, drop_last=False)

    model = timm.create_model("resnet18", pretrained=True, num_classes=4).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    for ep in range(1, epochs+1):
        model.train()
        run = 0.0
        for x, y in dl:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            run += loss.item()
        print(f"ResNet fold train ep {ep}/{epochs} - loss: {run/len(dl):.4f}")

    return model

## ResNet18 inference: slide-level probabilities

Tile softmax probabilities are averaged per slide. Averaging rather than
max-pooling matters at this dataset size: a max over tiles lets one confident
outlier tile decide the slide.


In [10]:
@torch.no_grad()
def infer_resnet_slide_probs(model, slide_list, bs=128):
    model.eval()
    out = {}
    for sid in slide_list:
        idxs = slide_to_idxs[sid]
        ds = TileDataset(train_tiles, labels_int=None, idxs=idxs, train_aug=False)
        dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)

        probs_all = []
        for x in dl:
            x = x.to(device, non_blocking=True)
            logits = model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs_all.append(probs)
        probs_all = np.concatenate(probs_all, axis=0)
        out[sid] = probs_all.mean(axis=0)
    return out

@torch.no_grad()
def infer_resnet_slide_probs_test(model, test_tiles_uint8, test_slide_ids, bs=128):
    model.eval()
    test_slide_to_idxs = defaultdict(list)
    for i, sid in enumerate(test_slide_ids):
        test_slide_to_idxs[sid].append(i)
    slide_list = sorted(test_slide_to_idxs.keys())

    out = {}
    for sid in slide_list:
        idxs = test_slide_to_idxs[sid]
        ds = TileDataset(test_tiles_uint8, labels_int=None, idxs=idxs, train_aug=False)
        dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)

        probs_all = []
        for x in dl:
            x = x.to(device, non_blocking=True)
            logits = model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs_all.append(probs)
        probs_all = np.concatenate(probs_all, axis=0)
        out[sid] = probs_all.mean(axis=0)
    return out

## Hugging Face authentication

UNI is a gated model on the Hugging Face Hub; downloading the weights requires an
account that has accepted its terms.


In [11]:
from huggingface_hub import login

login()

## Loading the UNI backbone

UNI is a Vision Transformer pretrained on roughly 100M histopathology images. The
backbone stays frozen throughout — only a small classifier head is trained on top
of its embeddings.


In [12]:
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from PIL import Image

uni_backbone = timm.create_model(
    "hf-hub:MahmoodLab/uni",
    pretrained=True,
    init_values=1e-5,
    dynamic_img_size=True,
    num_classes=0
).to(device)
uni_backbone.eval()

uni_transform = create_transform(**resolve_data_config(uni_backbone.pretrained_cfg, model=uni_backbone))

print("UNI embed dim:", getattr(uni_backbone, "num_features", "unknown"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning:
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

UNI embed dim: 1024


## Caching the UNI embeddings

The frozen backbone is run once over every tile and the 1024-dimensional
embeddings are cached to disk. Every later fold trains its classifier head on
these cached vectors, which turns 5-fold cross-validation over a ViT into a few
minutes of MLP fitting.


In [13]:
@torch.no_grad()
def extract_uni_embeddings_all(tiles_uint8_hwc, bs=64):
    uni_backbone.eval()
    N = len(tiles_uint8_hwc)
    feats = np.zeros((N, 1024), dtype=np.float16)

    for i in range(0, N, bs):
        batch = tiles_uint8_hwc[i:i+bs]
        imgs = [uni_transform(Image.fromarray(t)) for t in batch]
        x = torch.stack(imgs, dim=0).to(device, non_blocking=True)
        f = uni_backbone(x)  # (bs,1024) float
        feats[i:i+len(batch)] = f.detach().cpu().numpy().astype(np.float16)

    return feats

UNI_TRAIN_FEATS_PATH = "uni_feats_train_fp16.npy"
UNI_TEST_FEATS_PATH  = "uni_feats_test_fp16.npy"

if os.path.exists(UNI_TRAIN_FEATS_PATH):
    uni_train_feats = np.load(UNI_TRAIN_FEATS_PATH, mmap_mode="r")
    print("loaded cached:", UNI_TRAIN_FEATS_PATH, uni_train_feats.shape, uni_train_feats.dtype)
else:
    uni_train_feats = extract_uni_embeddings_all(train_tiles, bs=64)
    np.save(UNI_TRAIN_FEATS_PATH, uni_train_feats)
    print("saved:", UNI_TRAIN_FEATS_PATH, uni_train_feats.shape, uni_train_feats.dtype)

if os.path.exists(UNI_TEST_FEATS_PATH):
    uni_test_feats = np.load(UNI_TEST_FEATS_PATH, mmap_mode="r")
    print("loaded cached:", UNI_TEST_FEATS_PATH, uni_test_feats.shape, uni_test_feats.dtype)
else:
    uni_test_feats = extract_uni_embeddings_all(test_tiles, bs=64)
    np.save(UNI_TEST_FEATS_PATH, uni_test_feats)
    print("saved:", UNI_TEST_FEATS_PATH, uni_test_feats.shape, uni_test_feats.dtype)

saved: uni_feats_train_fp16.npy (3392, 1024) float16
saved: uni_feats_test_fp16.npy (2547, 1024) float16


## UNI classifier head

`Linear(1024 -> 256) -> ReLU -> Dropout(0.4) -> Linear(256 -> 4)`. Deliberately
small: the representation is doing the work, and a larger head on 627 slides
would only overfit it.


In [14]:
class UNIClassifier(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 4)
        )
    def forward(self, x):
        return self.net(x)

## UNI tile dataset + training on a fold


In [15]:
class UniFeatTileDataset(Dataset):
    def __init__(self, feats_fp16, labels_int, idxs):
        self.X = feats_fp16[idxs]  # (n,1024) fp16/float
        self.y = labels_int[idxs].astype(np.int64)
        self.idxs = idxs

    def __len__(self):
        return len(self.idxs)

    def __getitem__(self, i):
        x = torch.tensor(self.X[i], dtype=torch.float32)
        y = torch.tensor(int(self.y[i]), dtype=torch.long)
        return x, y

def train_uni_classifier_on_fold(train_tile_idxs, epochs=25, bs=256, lr=1e-3, wd=1e-4, dropout=0.4, label_smoothing=0.0):
    y_fold = train_labels[train_tile_idxs].astype(np.int64)
    weights = compute_class_weights_from_tile_labels(y_fold, n_classes=4)

    ds = UniFeatTileDataset(uni_train_feats, train_labels, idxs=train_tile_idxs)
    dl = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True, drop_last=False)

    clf = UNIClassifier(dropout=dropout).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=wd)

    for ep in range(1, epochs+1):
        clf.train()
        run = 0.0
        for x, y in dl:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = clf(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            run += loss.item()
        print(f"UNI clf fold train ep {ep}/{epochs} - loss: {run/len(dl):.4f}")

    return clf

## UNI inference: slide mean probs (train/val and test)


In [16]:
@torch.no_grad()
def infer_uni_slide_probs(clf, slide_list, bs=1024):
    clf.eval()
    out = {}
    for sid in slide_list:
        idxs = slide_to_idxs[sid]
        X = torch.tensor(uni_train_feats[idxs], dtype=torch.float32)  # (n,1024)
        probs_all = []
        for i in range(0, X.size(0), bs):
            x = X[i:i+bs].to(device, non_blocking=True)
            logits = clf(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs_all.append(probs)
        probs_all = np.concatenate(probs_all, axis=0)
        out[sid] = probs_all.mean(axis=0)
    return out

@torch.no_grad()
def infer_uni_slide_probs_test(clf, uni_test_feats_fp16, test_slide_ids, bs=1024):
    clf.eval()
    test_slide_to_idxs = defaultdict(list)
    for i, sid in enumerate(test_slide_ids):
        test_slide_to_idxs[sid].append(i)
    slide_list = sorted(test_slide_to_idxs.keys())

    out = {}
    for sid in slide_list:
        idxs = test_slide_to_idxs[sid]
        X = torch.tensor(uni_test_feats_fp16[idxs], dtype=torch.float32)
        probs_all = []
        for i in range(0, X.size(0), bs):
            x = X[i:i+bs].to(device, non_blocking=True)
            logits = clf(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs_all.append(probs)
        probs_all = np.concatenate(probs_all, axis=0)
        out[sid] = probs_all.mean(axis=0)
    return out

## Slide-level stratified 5-fold split

Stratified on the slide label and grouped by slide. Both branches see exactly the
same folds, which is what makes their out-of-fold probabilities directly
comparable and therefore safe to mix.


In [17]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

folds = []
for f, (tr_idx, va_idx) in enumerate(skf.split(train_slides, train_slide_y)):
    tr_slides = train_slides[tr_idx].tolist()
    va_slides = train_slides[va_idx].tolist()
    folds.append((tr_slides, va_slides))
    print(f"fold {f}: train_slides={len(tr_slides)} val_slides={len(va_slides)}")

fold 0: train_slides=501 val_slides=126
fold 1: train_slides=501 val_slides=126
fold 2: train_slides=502 val_slides=125
fold 3: train_slides=502 val_slides=125
fold 4: train_slides=502 val_slides=125


## Cross-validation loop

For each fold: train ResNet18 on the fold's tiles, train the UNI head on the same
slides' cached embeddings, and predict the held-out slides with both. The result
is one out-of-fold probability vector per slide per branch, from models that
never saw that slide.


In [20]:
# Hyperparams (keep consistent with your best runs)
RESNET_CFG = dict(epochs=15, bs=64, lr=1e-4, wd=1e-2, label_smoothing=0.1)
UNI_CFG    = dict(epochs=25, bs=256, lr=1e-3, wd=1e-4, dropout=0.4, label_smoothing=0.0)

oof_resnet = {}
oof_uni = {}
oof_y = {}

for fold_id, (tr_slides, va_slides) in enumerate(folds):
    print("\n" + "="*60)
    print("FOLD", fold_id)
    print("="*60)

    train_tile_idxs = []
    for sid in tr_slides:
        train_tile_idxs.extend(slide_to_idxs[sid])
    train_tile_idxs = np.array(train_tile_idxs, dtype=np.int64)

    # ResNet train
    resnet = train_resnet18_on_fold(train_tile_idxs, train_labels, **RESNET_CFG)

    # ResNet val inference
    val_resnet_probs = infer_resnet_slide_probs(resnet, va_slides, bs=128)
    for sid in va_slides:
        oof_resnet[sid] = val_resnet_probs[sid]

    # UNI classifier train
    uni_clf = train_uni_classifier_on_fold(train_tile_idxs, **UNI_CFG)

    # UNI val inference
    val_uni_probs = infer_uni_slide_probs(uni_clf, va_slides, bs=1024)
    for sid in va_slides:
        oof_uni[sid] = val_uni_probs[sid]
        oof_y[sid] = slide_to_label[sid]

    # Optional: free VRAM
    del resnet, uni_clf
    torch.cuda.empty_cache()

print("\nOOF collected:", len(oof_y), "slides")


FOLD 0
ResNet fold train ep 1/15 - loss: 1.4147
ResNet fold train ep 2/15 - loss: 1.3697
ResNet fold train ep 3/15 - loss: 1.3325
ResNet fold train ep 4/15 - loss: 1.2908
ResNet fold train ep 5/15 - loss: 1.2536
ResNet fold train ep 6/15 - loss: 1.2135
ResNet fold train ep 7/15 - loss: 1.1813
ResNet fold train ep 8/15 - loss: 1.1531
ResNet fold train ep 9/15 - loss: 1.1171
ResNet fold train ep 10/15 - loss: 1.0869
ResNet fold train ep 11/15 - loss: 1.0676
ResNet fold train ep 12/15 - loss: 1.0430
ResNet fold train ep 13/15 - loss: 1.0074
ResNet fold train ep 14/15 - loss: 0.9797
ResNet fold train ep 15/15 - loss: 0.9626
UNI clf fold train ep 1/25 - loss: 1.3045
UNI clf fold train ep 2/25 - loss: 1.1186
UNI clf fold train ep 3/25 - loss: 1.0138
UNI clf fold train ep 4/25 - loss: 0.9543
UNI clf fold train ep 5/25 - loss: 0.8890
UNI clf fold train ep 6/25 - loss: 0.8226
UNI clf fold train ep 7/25 - loss: 0.7658
UNI clf fold train ep 8/25 - loss: 0.7198
UNI clf fold train ep 9/25 - loss: 

## Saving the out-of-fold probabilities

Written to `oof_resnet_uni_5fold.npz`. This file is committed to the repository
as `artifacts/oof_slide_probs_5fold.npz`: it is what makes the weight search and
the ablation reproducible without a GPU.


In [21]:
# === SAVE OOF ARTIFACTS ===

oof_sids = sorted(oof_y.keys())

oof_resnet_arr = np.stack([oof_resnet[s] for s in oof_sids], axis=0)  # (N,4)
oof_uni_arr    = np.stack([oof_uni[s] for s in oof_sids], axis=0)     # (N,4)
oof_labels_arr = np.array([oof_y[s] for s in oof_sids], dtype=np.int64)

np.savez(
    "oof_resnet_uni_5fold.npz",
    slide_ids=np.array(oof_sids),
    probs_resnet=oof_resnet_arr,
    probs_uni=oof_uni_arr,
    labels=oof_labels_arr
)

print("Saved OOF to oof_resnet_uni_5fold.npz")

Saved OOF to oof_resnet_uni_5fold.npz


## Sanity checks for OOF shapes + baseline fold-free scores


In [22]:
oof_sids = sorted(oof_y.keys())
Y_true = np.array([oof_y[s] for s in oof_sids], dtype=np.int64)

P_res = np.stack([oof_resnet[s] for s in oof_sids], axis=0)
P_uni = np.stack([oof_uni[s] for s in oof_sids], axis=0)

print("OOF shapes:", P_res.shape, P_uni.shape, Y_true.shape)

pred_res = P_res.argmax(axis=1)
pred_uni = P_uni.argmax(axis=1)

print("OOF ResNet macro F1:", f1_score(Y_true, pred_res, average="macro"))
print("OOF UNI    macro F1:", f1_score(Y_true, pred_uni, average="macro"))

OOF shapes: (627, 4) (627, 4) (627,)
OOF ResNet macro F1: 0.4248271992049081
OOF UNI    macro F1: 0.4723133356439907


## Tuning the mixing weight

Grid search over `alpha` in the ensemble `alpha * P_ResNet + (1 - alpha) * P_UNI`,
scored on out-of-fold predictions. Tuning on out-of-fold rather than in-fold
probabilities is what keeps the chosen weight from simply rewarding whichever
branch overfits harder.


In [23]:
def tune_alpha(P_res, P_uni, Y_true, grid=np.linspace(0,1,101)):
    best = (-1.0, None)
    for a in grid:
        P = a * P_res + (1.0 - a) * P_uni
        pred = P.argmax(axis=1)
        f1 = f1_score(Y_true, pred, average="macro")
        if f1 > best[0]:
            best = (f1, float(a))
    return best

best_f1, best_alpha = tune_alpha(P_res, P_uni, Y_true, np.linspace(0,1,101))
print("best coarse:", best_f1, "alpha:", best_alpha)

# refine around best_alpha
lo = max(0.0, best_alpha - 0.05)
hi = min(1.0, best_alpha + 0.05)
fine_grid = np.linspace(lo, hi, 101)
best_f1_f, best_alpha_f = tune_alpha(P_res, P_uni, Y_true, fine_grid)
print("best refined:", best_f1_f, "alpha:", best_alpha_f)

BEST_ALPHA = best_alpha_f

best coarse: 0.4746889157514982 alpha: 0.05
best refined: 0.4746889157514982 alpha: 0.041


## OOF ensemble score + confusion matrix


In [24]:
P_ens = BEST_ALPHA * P_res + (1.0 - BEST_ALPHA) * P_uni
pred_ens = P_ens.argmax(axis=1)

print("OOF ensemble macro F1:", f1_score(Y_true, pred_ens, average="macro"))
print("Confusion matrix (OOF):")
print(confusion_matrix(Y_true, pred_ens))

OOF ensemble macro F1: 0.4746889157514982
Confusion matrix (OOF):
[[ 90  51  25   8]
 [ 52 109  44  14]
 [ 33  51  60  14]
 [  6  17  15  38]]


## Refitting on all training slides

The cross-validated models exist to produce honest out-of-fold probabilities.
The models that predict the test set are retrained here from scratch on every
training slide.


In [25]:
all_train_tile_idxs = np.arange(len(train_tiles), dtype=np.int64)
print("all train tiles:", len(all_train_tile_idxs))

all train tiles: 3392


## Full training ResNet18 + save


In [26]:
FULL_RESNET_PATH = "resnet18_full_bestalpha.pth"

resnet_full = train_resnet18_on_fold(all_train_tile_idxs, train_labels, **RESNET_CFG)
torch.save(resnet_full.state_dict(), FULL_RESNET_PATH)
print("saved:", FULL_RESNET_PATH)

ResNet fold train ep 1/15 - loss: 1.4030
ResNet fold train ep 2/15 - loss: 1.3520
ResNet fold train ep 3/15 - loss: 1.3115
ResNet fold train ep 4/15 - loss: 1.2618
ResNet fold train ep 5/15 - loss: 1.2229
ResNet fold train ep 6/15 - loss: 1.1997
ResNet fold train ep 7/15 - loss: 1.1635
ResNet fold train ep 8/15 - loss: 1.1281
ResNet fold train ep 9/15 - loss: 1.1004
ResNet fold train ep 10/15 - loss: 1.0682
ResNet fold train ep 11/15 - loss: 1.0363
ResNet fold train ep 12/15 - loss: 1.0083
ResNet fold train ep 13/15 - loss: 0.9823
ResNet fold train ep 14/15 - loss: 0.9631
ResNet fold train ep 15/15 - loss: 0.9335
saved: resnet18_full_bestalpha.pth


## Full training UNI classifier + save


In [27]:
FULL_UNI_CLF_PATH = "uni_clf_full_bestalpha.pth"

uni_clf_full = train_uni_classifier_on_fold(all_train_tile_idxs, **UNI_CFG)
torch.save(uni_clf_full.state_dict(), FULL_UNI_CLF_PATH)
print("saved:", FULL_UNI_CLF_PATH)

UNI clf fold train ep 1/25 - loss: 1.2792
UNI clf fold train ep 2/25 - loss: 1.1154
UNI clf fold train ep 3/25 - loss: 1.0270
UNI clf fold train ep 4/25 - loss: 0.9400
UNI clf fold train ep 5/25 - loss: 0.8944
UNI clf fold train ep 6/25 - loss: 0.8253
UNI clf fold train ep 7/25 - loss: 0.7925
UNI clf fold train ep 8/25 - loss: 0.7226
UNI clf fold train ep 9/25 - loss: 0.6878
UNI clf fold train ep 10/25 - loss: 0.6337
UNI clf fold train ep 11/25 - loss: 0.6073
UNI clf fold train ep 12/25 - loss: 0.5702
UNI clf fold train ep 13/25 - loss: 0.5395
UNI clf fold train ep 14/25 - loss: 0.5234
UNI clf fold train ep 15/25 - loss: 0.4758
UNI clf fold train ep 16/25 - loss: 0.4524
UNI clf fold train ep 17/25 - loss: 0.4422
UNI clf fold train ep 18/25 - loss: 0.4099
UNI clf fold train ep 19/25 - loss: 0.3934
UNI clf fold train ep 20/25 - loss: 0.3691
UNI clf fold train ep 21/25 - loss: 0.3492
UNI clf fold train ep 22/25 - loss: 0.3320
UNI clf fold train ep 23/25 - loss: 0.3072
UNI clf fold train e

## Test inference (slide probs) for both models


In [28]:
# ResNet test slide probs
resnet_test_probs = infer_resnet_slide_probs_test(resnet_full, test_tiles, test_slide_ids, bs=128)

# UNI test slide probs
uni_test_probs = infer_uni_slide_probs_test(uni_clf_full, uni_test_feats, test_slide_ids, bs=2048)

print("test slides (resnet):", len(resnet_test_probs))
print("test slides (uni):   ", len(uni_test_probs))

test slides (resnet): 477
test slides (uni):    477


## Ensemble with BEST_ALPHA + submission.csv


In [29]:
# Ensure same slide order
test_unique_slides = sorted(set(test_slide_ids.tolist()))

rows = []
for sid in test_unique_slides:
    p_res = resnet_test_probs[sid]
    p_uni = uni_test_probs[sid]
    p = BEST_ALPHA * p_res + (1.0 - BEST_ALPHA) * p_uni
    pred = int(np.argmax(p))
    rows.append((sid, IDX_TO_LABEL[pred]))

sub = pd.DataFrame(rows, columns=["sample_index", "label"])
sub = sub.sort_values("sample_index").reset_index(drop=True)

SUB_PATH = "submission_1312.csv"
sub.to_csv(SUB_PATH, index=False)
print("saved:", SUB_PATH)
sub.head()

saved: submission_1312.csv


,sample_index,label
0,img_0000.png,Luminal B
1,img_0001.png,Luminal B
2,img_0002.png,HER2(+)
3,img_0003.png,Luminal B
4,img_0004.png,Luminal A


In [30]:
def validate_submission(csv_path, start_id=0, end_id=476):
    df = pd.read_csv(csv_path)
    assert set(df.columns) == {"sample_index", "label"}

    expected = [f"img_{i:04d}.png" for i in range(start_id, end_id+1)]
    got = df["sample_index"].tolist()

    missing = sorted(set(expected) - set(got))
    extra = sorted(set(got) - set(expected))
    dup = df[df.duplicated("sample_index", keep=False)]

    valid_labels = set(IDX_TO_LABEL.values())
    bad_lab = df[~df["label"].isin(valid_labels)]

    print("rows:", len(df))
    print("missing:", len(missing), "extra:", len(extra), "dups:", len(dup), "bad_labels:", len(bad_lab))

    if missing[:5]:
        print("missing ex:", missing[:5])
    if extra[:5]:
        print("extra ex:", extra[:5])
    if len(dup):
        print("dup ex:\n", dup.head())
    if len(bad_lab):
        print("bad labels ex:\n", bad_lab.head())

    if (len(missing)==0) and (len(extra)==0) and (len(dup)==0) and (len(bad_lab)==0):
        print("OK")

validate_submission(SUB_PATH, 0, 476)

rows: 477
missing: 0 extra: 0 dups: 0 bad_labels: 0
OK
